# SR-LPNE — Train on SM90, Test on Observations

Notebook version of `Train_Emulator_obs_SR-LPNE.py`. Same code, split into cells so each stage
(data loading, training, evaluation) can be run and inspected independently, with all figures
shown inline instead of only written to disk.

The workflow: the emulator is trained and validated on clean, synthetic SM90 model output
(70% train / 30% validation), then tested against the interpolated observational record
(`interpolateddata.csv`) that stands in for real ice-core proxies (δ18O, CO2, CH4) paired with
insolation.


## Imports

In [ ]:
%matplotlib inline 
%load_ext autoreload
%autoreload 2

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42


## Model definition

Helper functions, the CNN architecture, the custom rollout-training `RolloutEmulator` model,
and the physical-attractor checkpoint callback that only saves weights when the C₂ constraint
and lag-1 ACF causality check both pass.

In [ ]:
def set_seeds(seed=seed):
    # set random see for tensorflow, keras, numpy etc.
    tf.keras.utils.set_random_seed(seed)
    # only relevant if calculation is done on GPU
    tf.config.experimental.enable_op_determinism()

def create_windows(data, window_size, rollout_length):
    X_list, y_list, Insol_rollout_list = [], [], []
    for i in range(len(data) - window_size - rollout_length):
        X_list.append(data[i:i + window_size,:])
        y_list.append(data[i + window_size: i + window_size + rollout_length, :-1])
        Insol_rollout_list.append(data[i + window_size: i + window_size + rollout_length, -1])

    return np.array(X_list), np.array(y_list), np.array(Insol_rollout_list)


In [ ]:
def CNN_model(window_size, n_features):
    """
    Convolutional Neural Network architecture to be used for the SR-LPNE model
    --------------------------------------------------------------------------

    window_size - input window size
    n_features - number of input variables (here set to 4 for X,Y,Z,Insol

    """
    # layer names are pinned explicitly (matching Keras's own default names for a
    # session's first CNN_model build) so save_weights()/load_weights() - which
    # match variables by layer name - keep working no matter how many times this
    # function has already been called in the current kernel session
    # 0 parameters
    inp = layers.Input(shape=(window_size, n_features))

    x = layers.GaussianNoise(0.05, name="gaussian_noise")(inp) #increased from 0.05

    # 16 filters and 5 kernel size
    x = layers.Conv1D(filters = params['conv filters'],
                      kernel_size = params['conv kernel'], # changed from 5 to 50 for high resolution
                      activation = 'swish',
                      padding = "causal",
                      name = "conv1d")(x)

    x = layers.LayerNormalization(name="layer_normalization")(x)

    # extract last tensor from convolutional layer output
    x = layers.Lambda(lambda x: x[:, -1, :], name="lambda")(x)

    x = layers.Dense(units = params['dense unit'],
                     activation="swish",
                     name = "dense")(x)

    # dropout layer to reduce overfitting; dropout rate is a hyperparameter
    x = layers.Dropout(rate=params['dropout rate'], name="dropout")(x)

    # output layer, returns the vector [\Delta_X,\Delta_Y,\Delta_Z]
    out = layers.Dense(units = 3,
                       activation = 'linear',
                       kernel_initializer = 'zeros',
                       bias_initializer = 'zeros',
                       use_bias=True,
                       name = "dense_1")(x)

    # physical constraint: scaling by maximum incerement in data
    # WARNING: NEEDS tO BE ADJUSTED FOR DIFFERENT DATA SETS
    #out = out * tf.constant([0.47893476,0.78733971,0.21167531], dtype=tf.float32)

    return Model(inputs = inp,outputs = out)


In [ ]:
class RolloutEmulator(tf.keras.Model):
    """
    custom keras model class for rollout training, validation and prediction

    Loss: Harmonized MSE (hMSE) computed on a multi-level (default 2-level) Haar
    DWT of the rollout trajectory, per "The Procrustean Bed of Time Series"
    (arXiv:2512.18610), Eq. 21-23. The DWT is implemented with tf.nn.conv1d using
    fixed (non-trainable) Haar filter taps so it stays differentiable under
    GradientTape - PyWavelets operates outside the TF graph and would break
    gradient flow. Multi-level decomposition is the standard recursive "pyramid
    algorithm": each level decomposes the previous level's approximation into a
    new, shorter approximation plus a detail band; every level's detail band is
    kept, and only the final (coarsest) approximation is kept from the last level.
    """
    def __init__(self,brain,rollout_length,vali_rollout_length,window_size,n_features,
                 L_2_weight=1.0,dwt_gamma=0.5,dwt_beta=0.3,dwt_eps=1e-6,dwt_levels=3):
        """
        initializer method for the rollout trainig class
        ------------------------------------------------

        brain               - model to train via rollout training
        rollout_length      - training rollout length
        vali_rollout_length - validation rollout length
        L_2_weight          - weight on the harmonized MSE loss (default 1.0)
        dwt_gamma           - Eq. 21 gamma: balances the flat vs. relative-error term (paper default 0.5)
        dwt_beta            - Eq. 23 beta: EMA smoothing factor for the historical-magnitude tracker (paper default 0.3)
        dwt_eps             - Eq. 21 epsilon: minimum threshold preventing weight explosion
        dwt_levels           - number of recursive DWT decomposition levels (default 2)
        noise_level         - standard deviation of normally distributed noise used for noise injection during training
        """

        super(RolloutEmulator, self).__init__()
        self.brain = brain
        self.rollout_length = rollout_length
        self.vali_rollout_length = vali_rollout_length
        self.L_2_weight = L_2_weight
        self.noise_level = 1e-2 # change to zero at some point

        self.brain.build((None, window_size, n_features))

        # --- Harmonized MSE (hMSE) setup --------------------------------------
        # Haar wavelet: orthonormal 2-tap low-pass (approximation) / high-pass (detail)
        # filters, shaped (filter_width, in_channels=1, out_channels=1) for conv1d.
        self.dwt_low  = tf.constant([[[1/np.sqrt(2)]], [[ 1/np.sqrt(2)]]], dtype=tf.float32)
        self.dwt_high = tf.constant([[[1/np.sqrt(2)]], [[-1/np.sqrt(2)]]], dtype=tf.float32)

        self.dwt_gamma = dwt_gamma
        self.dwt_beta = dwt_beta
        self.dwt_eps = dwt_eps
        self.dwt_levels = dwt_levels

        # historical-magnitude EMA state (f_bar in Eq. 23), one tracker per DWT
        # coefficient group (each level's detail band, plus the final coarsest
        # approximation), one value per (coefficient index, state variable).
        # train_step and vali_step roll out to different lengths, so each needs
        # its own set of trackers sized to its own coefficient-group lengths.
        # Haar DWT with stride 2 / VALID padding on length L gives L//2
        # coefficients per level (an odd L would floor-truncate the last sample -
        # both rollout lengths here are even at every level this goes to, so this
        # doesn't come up).
        train_group_lengths = self._dwt_multilevel_lengths(self.rollout_length, self.dwt_levels)
        vali_group_lengths = self._dwt_multilevel_lengths(self.vali_rollout_length, self.dwt_levels)

        self.fbar_train = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_train_{i}")
                            for i, L in enumerate(train_group_lengths)]
        self.fbar_vali = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_vali_{i}")
                           for i, L in enumerate(vali_group_lengths)]

    # returns number of (trainable) variables/ size of the neural network
    @property
    def trainable_variables(self):
        return self.brain.trainable_variables

    # call to model to predict dependent on the inputs (training set to None)
    def call(self, inputs, training=None):
        return self.brain(inputs, training=training)

    def _dwt(self, x, length):
        """
        Shallow (1-level) Haar DWT applied independently to each of the 3 state
        channels (X, Y, Z), via tf.nn.conv1d with the channel dimension folded
        into the batch dimension so channels aren't mixed together.

        x: (batch, length, 3) -> cA, cD: (batch, length//2, 3)
        """
        batch = tf.shape(x)[0]
        coeff_len = length // 2

        x_ch = tf.transpose(x, [0, 2, 1])              # (batch, 3, length)
        x_ch = tf.reshape(x_ch, [-1, length, 1])        # (batch*3, length, 1)

        cA = tf.nn.conv1d(x_ch, self.dwt_low,  stride=2, padding="VALID")  # (batch*3, coeff_len, 1)
        cD = tf.nn.conv1d(x_ch, self.dwt_high, stride=2, padding="VALID")

        cA = tf.transpose(tf.reshape(cA, [batch, 3, coeff_len]), [0, 2, 1])  # (batch, coeff_len, 3)
        cD = tf.transpose(tf.reshape(cD, [batch, 3, coeff_len]), [0, 2, 1])
        return cA, cD

    def _dwt_multilevel(self, x, length, levels):
        """
        Recursive multi-level (Mallat pyramid) Haar DWT, built on top of the
        single-level _dwt above: at each level, the current approximation is
        decomposed into a new, shorter approximation plus a detail band; the
        detail band is kept and the approximation is fed into the next level.

        x: (batch, length, 3)
        returns: a list of (batch, len_i, 3) tensors, ordered
                 [cD_level1, cD_level2, ..., cD_levelN, cA_levelN]
                 i.e. detail coefficients from finest to coarsest scale, followed
                 by the final (coarsest) approximation.
        """
        coeffs = []
        current, current_length = x, length
        for _ in range(levels):
            cA, cD = self._dwt(current, current_length)
            coeffs.append(cD)
            current, current_length = cA, current_length // 2
        coeffs.append(current)  # final coarse approximation
        return coeffs

    @staticmethod
    def _dwt_multilevel_lengths(length, levels):
        """Coefficient-group lengths for _dwt_multilevel, in the same order it returns them."""
        lengths = []
        current_length = length
        for _ in range(levels):
            current_length = current_length // 2
            lengths.append(current_length)
        lengths.append(current_length)  # final approximation shares the last level's length
        return lengths

    def _harmonized_mse(self, true_seq, pred_seq, length, fbar_list, update):
        """
        Harmonized MSE (Eq. 21) computed on a multi-level Haar DWT of true_seq/pred_seq.

        true_seq, pred_seq - (batch, length, 3) ground-truth / predicted trajectories
        fbar_list           - list of (len_i, 3) EMA trackers, one per DWT coefficient
                               group (see _dwt_multilevel for the group ordering)
        update               - whether to update fbar_list with this batch (Eq. 23)
        """
        true_groups = self._dwt_multilevel(true_seq, length, self.dwt_levels)
        pred_groups = self._dwt_multilevel(pred_seq, length, self.dwt_levels)

        loss = 0.0
        for true_g, pred_g, fbar in zip(true_groups, pred_groups, fbar_list):
            if update:
                batch_mag = tf.reduce_mean(tf.abs(true_g), axis=0)  # (len_i, 3)
                fbar.assign(self.dwt_beta * fbar + (1 - self.dwt_beta) * batch_mag)

            weight = 1.0 + self.dwt_gamma / (fbar + self.dwt_eps)  # (len_i, 3)
            loss += tf.reduce_mean(weight * tf.square(true_g - pred_g))

        return loss

    # heart of the actual training step
    @tf.function
    def train_step(self,data):

        # as you can see here, the training step will recieve data in a tuple of (features,data), which is equivalent to the data you pass to the
        # x= ... and y= .. argument in the keras.Model.fit(...) method
        (features,y) = data

        # features itself contains the X_init values being the values of the state space (X,Y,Z) and the insolation forcing in the input window w at index 0
        # at index 1 the "future" insolation forcing aka. the insolation during the rollout is passed with shape (rollout_length,1)
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        # scale augmentation to randomly rescale state space feature and target trajectories
        scale = tf.random.uniform([batch_size,1,1], 0.8, 1.2)

        # apply scaling to (X,Y,Z) in the input window
        X_init_scaled = tf.concat([X_init[:, :, 0:3] * scale, X_init[:, :, 3:]], axis=-1)
        y_scaled = y * scale

        # tell tensorflow to create the context manager of the variables created in the with statement and tracks them to calculate the gradients
        with tf.GradientTape() as tape:

            # passing input window as first window to use in rollout training
            current_window = X_init_scaled

            # calculate ground truth trajectory
            true_path = tf.concat([X_init_scaled[:, -1:, 0:3], y_scaled], axis=1)

            all_paths = []

            # actual rollout training
            for i in range(self.rollout_length):

                # current window (first step this is just the input window) with added normally distributed (input) noise
                noisy_window = current_window + tf.random.normal(tf.shape(current_window), stddev=self.noise_level)

                # generating increment predictions (forward-pass)
                d_prediction = self.brain(noisy_window, training=True)

                # adding predicted increment to last entry in the crurrent windoe
                x_now = current_window[:, -1, 0:3]
                prediction_abs = x_now + d_prediction

                all_paths.append(prediction_abs)

                # adding additional output noise
                prediction_abs += tf.random.normal(tf.shape(prediction_abs), stddev=self.noise_level)

                next_step = tf.zeros((batch_size, 1, n_features))

                pred = tf.expand_dims(prediction_abs, axis=1)

                forc_true = future_forcing[:, i]

                # noise injection usedto prevent NN from remembering just the values, resulting in insolation just being used as a clock (safety measure)
                forc_noisy = forc_true + tf.random.normal(tf.shape(forc_true), stddev=self.noise_level)

                forc = tf.reshape(forc_noisy, (batch_size, 1, 1))
                next_step = tf.concat([pred,forc],axis=-1)

                # appending next predicted state and insolation forcig to the current window and moving window one time step forward
                current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

            # creating stack over rollout window for trajectory
            p_stack = tf.stack(all_paths, axis=1)

            # Harmonized MSE, computed on a shallow Haar DWT of the trajectory (Eq. 21)
            abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.rollout_length,
                                             self.fbar_train, update=True)
            total_loss = self.L_2_weight * abs_loss

            # plain point-to-point MSE, logged for comparison only - not part of total_loss,
            # so it has no effect on the gradients computed below
            pointwise_mse = tf.reduce_mean(tf.square(true_path[:,1:,:] - p_stack))

        # Backporpagation through the forward pass operations tracked by GradientTape()
        grads = tape.gradient(total_loss,self.trainable_variables)
        # weight update
        self.optimizer.apply_gradients(zip(grads,self.trainable_variables))

        return {
            "loss": total_loss,
            "abs": abs_loss,
            "mse": pointwise_mse}

    # test_step used in the validation process
    # essentially the same as the training step, only with training disabled and no backpropagation/ weights update
    def test_step(self,data):

        def calculate_lag1_acf(p_stack):
            """
            calculate the lag_1 auto-correlation function to ensure causality in the validation step is preserved
            --------------------------------------------------

            p_stack - predicted validation rollout trajetory
            """

            mu = tf.reduce_mean(p_stack, axis=1, keepdims=True)
            var = tf.math.reduce_variance(p_stack, axis=1, keepdims=True)

            p_centered = p_stack - mu
            t0 = p_centered[:, :-1, :]
            t1 = p_centered[:, 1:, :]

            acf_per_sample = tf.reduce_mean(t0 * t1, axis=1, keepdims=True) / (var + 1e-7)

            return tf.reduce_mean(acf_per_sample)


        (features,y) = data
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        current_window = X_init

        all_paths = []

        true_path = tf.concat([X_init[:, -1:, 0:3], y], axis=1) # [Batch, Rollout+1, 3]

        for i in range(self.vali_rollout_length):
            d_prediction = self.brain(current_window, training=False)

            x_now = current_window[:, -1, 0:3]
            prediction_abs = x_now + d_prediction

            all_paths.append(prediction_abs)

            next_step = tf.zeros((batch_size, 1, n_features))

            pred = tf.expand_dims(prediction_abs, axis=1)
            forc = tf.reshape(future_forcing[:, i], (batch_size, 1, 1))
            next_step = tf.concat([pred,forc],axis=-1)

            current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

        p_stack = tf.stack(all_paths, axis=1)

        # Harmonized MSE on the validation-length rollout - its own EMA tracker,
        # since the coefficient count differs from the training rollout length.
        # Updated here too (each validation pass) so its weighting is actually
        # adaptive rather than frozen at the initial f_bar=1 value; the paper
        # describes the EMA as updating "at each training iteration," so switch
        # update=False here if you want strictly training-only magnitude tracking.
        abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.vali_rollout_length,
                                         self.fbar_vali, update=True)

        # plain point-to-point MSE, logged for comparison only
        pointwise_mse = tf.reduce_mean(tf.square(true_path[:,1:,:] - p_stack))

        lag1_acf = calculate_lag1_acf(p_stack)

        # calculate standard deviation for ground truth and prediction in validation step
        y_std = tf.math.reduce_std(y,axis = 1)
        p_std = tf.math.reduce_std(p_stack, axis=1)

        # calculate C_2 constraint
        C_2 =  tf.reduce_mean(tf.abs(1.0 - (p_std / (y_std + 1e-6))))

        total_loss = self.L_2_weight * abs_loss

        return {
            "loss": total_loss,
            "abs": abs_loss,
            "mse": pointwise_mse,
            "lag1_acf": lag1_acf,
            "C_2": C_2}

In [ ]:
class PhysicalAttractorCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self,filepath="best_state_space_model.weights.h5",acf_threshold=0.97):
        """
        initializer method for C_2 constraint callback
        ------------------------------------------------

        filepath               - file path to save the best model weights
        acf_threshold          - threshold fro acf_function to accept model
        """
        super(PhysicalAttractorCheckpoint,self).__init__()
        self.filepath = filepath
        self.acf_threshold = acf_threshold
        self.best_C_2 = float('inf')
        self.best_epoch = 0
        self.patience = 0

    def on_epoch_end(self,epoch=0,logs=None):

        # retrieve current C_2 and lag_1 ACF values
        current_C_2 = logs.get("val_C_2")
        current_acf = logs.get("val_lag1_acf")

        if current_C_2 is None or current_acf is None:
            print("\nWarning: Physical metrics not found in logs. Check test_step return keys.")
            return

        # convert from string to float
        current_C_2 = float(current_C_2)
        current_acf = float(current_acf)

        # check if ACF criteria is meet
        if current_acf < self.acf_threshold:
            if (epoch + 1) % 5 == 0:
                print(f"\nEpoch {epoch+1}: Smoothness ({current_acf:.4f}) below threshold. Ignoring.")
            self.patience += 1
            if self.patience == 15:
                self.model.stop_training = True
                print(f"\nPatience ran out. (Should) Early stopping at epoch {epoch}: no physically-valid "
                      f"improvement in val_C_2 for {self.patience} epochs.")
            return

        # check if current model returns a lower C_2 constraint then previous model, if yes overwrite the current best model
        if current_C_2 < self.best_C_2:
            self.best_C_2 = current_C_2
            self.best_epoch = epoch
            self.model.save_weights(self.filepath)
            self.patience = 0
            print(f"\nEpoch {epoch+1}: NEW PHYSICAL BEST!")
            print(f" > C_2: {current_C_2:.4f}")
            print(f" > Lag-1 ACF:    {current_acf:.4f}")
            print(f" > Weights saved to {self.filepath}")
        else:
            self.patience += 1
            if self.patience == 15:
                self.model.stop_training = True
                print(f"\nPatience ran out. (Should) Early stopping at epoch {epoch}: no physically-valid "
                      f"improvement in val_C_2 for {self.patience} epochs.")


class WeightStatsLogger(tf.keras.callbacks.Callback):
    """Prints min/max/std of the harmonized-MSE weight tensors (1 + gamma/(fbar+eps))
    at the end of every epoch, for both the training- and validation-length trackers.
    Pools the weights across every DWT coefficient group (all decomposition levels)."""

    def _weight_stats(self, fbar_list):
        gamma, eps = self.model.dwt_gamma, self.model.dwt_eps
        weights = tf.concat([tf.reshape(1.0 + gamma / (fbar + eps), [-1]) for fbar in fbar_list], axis=0)
        return tf.reduce_min(weights), tf.reduce_max(weights), tf.math.reduce_std(weights)

    def on_epoch_end(self, epoch, logs=None):
        train_min, train_max, train_std = self._weight_stats(self.model.fbar_train)
        vali_min, vali_max, vali_std = self._weight_stats(self.model.fbar_vali)
        print(f" > hMSE weights (train): min={train_min:.4f}, max={train_max:.4f}, std={train_std:.4f}")
        print(f" > hMSE weights (vali):  min={vali_min:.4f}, max={vali_max:.4f}, std={vali_std:.4f}")


## Configuration

Random seed, plot styling, and the window/rollout sizes (all expressed as physical kyr spans,
divided by `timestep` to get the number of array steps — see the earlier discussion in this
project about keeping these calibrated to the actual sample spacing of the data).

In [ ]:
# ---- RUN CONFIGURATION ------------------------------------------------------
# Everything that defines a run lives here. The run tag - and therefore the folder
# Scripts/runs/<run_tag>/ where the weights + config (+ scaler) are saved - is derived
# automatically from these settings (see run_config.py), so changing the dataset,
# split, hyperparameters or seed can never overwrite another run's weights.
import sys, importlib
sys.path.insert(0, '/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Scripts')
import run_config as rc
importlib.reload(rc)

DATASET = '1461ka'      # '1243ka' or '1461ka'

cfg = rc.make_config(
    DATASET,
    seed=seed,
    params=None,         # None -> tuned params for DATASET; e.g. rc.DATASETS['1243ka']['params'] to swap
    test_frac=0.2,       # pass test_size=461 instead to reproduce the old hard-coded split
)
params = cfg['params']
paths = rc.run_paths(cfg['run_tag'])

print("run tag :", cfg['run_tag'])
print("weights :", paths['weights'])
print("params  :", params)


In [ ]:
# set random initilization seed
set_seeds(cfg['seed'])
plt.rcParams.update({'font.size': 20})

# window / rollout lengths are defined in kyr in run_config.make_config and converted
# to array steps there (window 50 kyr, training rollout 32 kyr, validation rollout 128 kyr)
timestep = cfg['timestep']  # kyr
window_size = cfg['window_size']
rollout_length = cfg['rollout_length']
rollout_length_validation = cfg['rollout_length_validation']

mappings = ["X","Y","Z"]

color_ls = [["#D6195E","#FFB600"],["#1E88E5","#6519C5"],["#05EFC5","#8DB1A7"]]


## Load and set up data

##### Now loading in and setting up the observational data for training and validating, and subsequently testing.

In [ ]:
# import interpolated observational data (dO18, CO2, MgCa, insolation vs. age)
# attempt 1 uses LR04-based data + EDC CO2 to train; uses LR04-based data to test
# ordered oldest -> youngest (row 0 = oldest age) by run_config.load_data
obs_data = rc.load_data(cfg)

# numpy arrays for each state variable, the insolation forcing, and the shared time axis
X = obs_data['dO18_LR04[pm]'].to_numpy(dtype=float)
Y = obs_data['CO2_Yamamoto[pm]'].to_numpy(dtype=float)
Z = obs_data['MgCa_Eld[mmol/mol]'].to_numpy(dtype=float)
R = obs_data['Insolation[W/m2]'].to_numpy(dtype=float)
time = obs_data['Age[kaBP]'].to_numpy(dtype=float)

print(f"data shape: {obs_data.shape}")

obs_data.head()


In [ ]:
flag = False

if flag:
    from climlab.solar.insolation import daily_insolation

    # Laskar et al. (2004) orbital solution: eccentricity, obliquity (rad), and
    # longitude of perihelion (rad, unwrapped/continuous) vs. time from J2000
    # (kyr). Covers t in [-51000, +21000], i.e. plenty of past AND future -
    # unlike climlab's bundled OrbitalTable (Berger & Loutre 1991), which only
    # goes back to -5000 kyr and has NO future (kyear > 0) values at all.
    la2004 = pd.read_csv(
        '/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/laskar2004/INSOL.LA2004.BTL.csv',
        comment='#', header=None, names=['t', 'ecc', 'obliq', 'varpi']
    )

    # target time axis: -1461 (1461 ka BP) to +10000 (10 Ma after present)
    start = -1*int(X.size)-1
    end = 10000
    years = np.linspace(start, end, end - start + 1)

    # interpolate orbital elements onto the target years. obliq/varpi are
    # interpolated in radians (varpi is left unwrapped in the source file) and
    # only converted to degrees + wrapped to [0, 360) afterward, to avoid
    # interpolation artifacts at the 0/360 boundary.
    ecc_interp = np.interp(years, la2004['t'], la2004['ecc'])
    obliquity_interp = np.degrees(np.interp(years, la2004['t'], la2004['obliq']))
    long_peri_interp = np.degrees(np.interp(years, la2004['t'], la2004['varpi'])) % 360

    days_of_year = np.arange(1, 366)
    tau = 0  # W/m2

    long_R = np.zeros_like(years)
    for i in tqdm(range(len(years)), desc="Computing ISI"):
        orb_i = {
            'ecc': ecc_interp[i],
            'obliquity': obliquity_interp[i],
            'long_peri': long_peri_interp[i],
        }
        W = daily_insolation(lat=65, day=days_of_year, orb=orb_i)
        J = np.sum(W[W >= tau] * 86400)  # joules/m2, summed over qualifying days
        long_R[i] = J / 1e9  # GJ/m2, matching Huybers' reported units

    long_R


In [ ]:
# split (shared with the MC notebooks via run_config.split_data, so both use the same convention):
#   test       = oldest test_frac of the record
#   (1-row buffer)
#   train      = oldest train_frac of the remaining train/validation set
#   validation = youngest remainder
splits = rc.split_data(obs_data, cfg)

test_size = splits['test_size']
test_data, test_time = splits['test'], splits['test_time']
trainval, trainval_time = splits['trainval'], splits['trainval_time']
fulldata, fulldata_time = splits['full'], splits['time']
train_data, validation_data = splits['train'], splits['validation']

print(rc.describe_split(splits))


In [ ]:
# fit standard scaler to train/valid data
train_scaler = rc.fit_scaler(splits)

# save config + scaler parameters into the run folder, so the MC notebooks can load
# exactly the scaler this model was trained with instead of refitting one
rc.save_run(cfg, train_scaler, splits)

# scale data with scaler
obs_train_data = train_scaler.transform(train_data)
test_data = train_scaler.transform(test_data)
obs_valid_data = train_scaler.transform(validation_data)

if flag:
    # long_R only has the insolation (R) column, so it can't go through
    # train_scaler.transform() directly (that expects all 4 fitted columns).
    long_R_scaled = (long_R - train_scaler.mean_[-1]) / train_scaler.scale_[-1]

X_train_obs, y_train_obs, Insol_train_obs = create_windows(obs_train_data, window_size, rollout_length)
X_valid_obs, y_valid_obs, Insol_valid_obs = create_windows(obs_valid_data, window_size, rollout_length_validation)

#np.save("test_set",test_data[:,:-1])
#np.save("obs_test_set",test_data[:,:-1])
#np.save("train_set",train_data[:,:-1])
#np.save("valid_set",validation_data[:,:-1])

#plt.plot(years, long_R_scaled)
#plt.show()


## Train NN

Define brain (NN architecture) and model (rollout emulator)

In [ ]:
brain = CNN_model(window_size=window_size, n_features=4)
brain.summary()

model = RolloutEmulator(
    brain,
    rollout_length=rollout_length,
    vali_rollout_length=rollout_length_validation,
    window_size=window_size,
    n_features=4,
    dwt_gamma=params['dwt gamma'],
    dwt_beta=params['dwt beta'],
    dwt_eps=params['dwt eps'],
    dwt_levels=params['dwt levels'],
)
model.noise_level = cfg['noise_level']  # noise injection std used in train_step
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=params['learning rate'], clipnorm=1.0))

dummy_X = tf.zeros((1, window_size, 4))
_ = model(dummy_X)

Subsequently, train on observational data.

In [ ]:
# commented out to try without SM90 pretraining

# lower LR than pretraining - refining an already-reasonable initialization,
# not learning from scratch; avoids blowing away the pretrained dynamics
#model.optimizer.learning_rate.assign(3e-4)

# if this exact config was trained before, keep its weights as model.prev.weights.h5
# rather than silently loading them later if this run never passes the ACF gate
if paths['weights'].exists():
    paths['weights'].replace(paths['prev_weights'])
    print(f"previous weights for this run tag moved to {paths['prev_weights'].name}")

phys_check = PhysicalAttractorCheckpoint(
    filepath=str(paths['weights']),
    acf_threshold=cfg['acf_threshold'],
)
weight_logger = WeightStatsLogger()

history = model.fit(
    [X_train_obs, Insol_train_obs],
    y_train_obs,
    validation_data=([X_valid_obs, Insol_valid_obs], y_valid_obs),
    epochs=cfg['epochs'],
    callbacks=[phys_check, weight_logger],
    batch_size=cfg['batch_size'],
    validation_batch_size=cfg['batch_size'],
    shuffle=True,
)




## Load best model and inspect training curves

In [ ]:
# return epoch, where lowest C_2 constrained model was found
best_ep = phys_check.best_epoch

if not paths['weights'].exists():
    raise RuntimeError("No epoch passed the ACF gate, so no weights were saved for this run.")

print(f"Loading best physical model for final evaluation from {paths['weights']} ...")
model.load_weights(str(paths['weights']))

Metrics = history.history
exportpath = paths['dir']   # figures for this run can go in its own run folder


In [ ]:
plt.gca()

# PLOTTING LOSS INFORMATION
fig,axs = plt.subplot_mosaic(("A;B;C"),figsize = (22,16))

axs['A'].plot(history.epoch,Metrics["abs"], label = 'Training Error')
axs['A'].plot(history.epoch,Metrics["val_abs"], label = 'Validation Error')
axs['A'].set_title("hMSE vs. Epochs", fontweight = "bold")
axs['A'].set_ylabel("$\mathcal{L}_{L_2}$")
axs['A'].set_xlabel("epochs")
axs['A'].grid(True, alpha=0.3)
axs['A'].legend()

axs['B'].plot(history.epoch,Metrics["val_C_2"],color = "lightseagreen", label="C$_2$")
axs['B'].plot(history.epoch,Metrics["val_lag1_acf"],color = "indigo", label="ACF")
axs['B'].set_title("Validation Metrics vs. Epochs", fontweight = "bold")
axs['B'].set_ylabel("Value")
axs['B'].set_xlabel("epochs")
axs['B'].legend(loc='center right')
axs['B'].grid(True, alpha=0.3)

axs['C'].plot(history.epoch,Metrics["mse"], label = 'Training Error')
axs['C'].plot(history.epoch,Metrics["val_mse"], label = 'Validation Error')
axs['C'].set_title("Point-to-Point MSE vs. Epochs", fontweight = "bold")
axs['C'].set_ylabel("MSE")
axs['C'].set_xlabel("epochs")
axs['C'].grid(True, alpha=0.3)
axs['C'].legend()

for key in axs:
    axs[key].axvline(x=best_ep, color='red', linestyle='--', alpha=0.6, label='Best Physical Model')

handles, labels = axs['A'].get_legend_handles_labels()

fig.legend(handles, labels,
           loc='lower center',
           bbox_to_anchor=(0.5, 0.02), # Centered, just above the very bottom
           ncol=3,
           fontsize=22,
           frameon=True, # Adding a frame can make it look more like a formal table
           facecolor='white',
           edgecolor='gray')

plt.tight_layout(rect=[0, 0.06, 1, 1])
#plt.savefig(f"{exportpath}/Loss_vs_Epochs_valid_rollout_longer.pdf",bbox_inches='tight')
plt.show()

## Roll the trained model forward on only the observational test set

First creating a rollout just on the test data (i.e. rolling out from 1.2-1ma bp). 

In [ ]:
# TEST SET 
@tf.function
def predict_step(w):
    return model(w, training=False)

ed = list(test_data[:window_size,:])

steps = test_data.shape[0]-window_size

# 3. Correct Loop
for t in tqdm(range(steps), desc="Test Forecasting"):

    nxt_step = np.zeros(4)
    # Grab the window directly from the main array to ensure it's always up to date
    current_window = np.array(ed[-window_size:])

    # Predict
    Input = np.expand_dims(current_window, axis=0)

    nxt_increments = predict_step(Input)

    nxt_step[0:3] = current_window[-1,0:3] + nxt_increments

    nxt_step[3] = test_data[window_size+t,-1]

    ed.append(nxt_step)

emulator_test_data = np.array(ed)



Subsequently, let the rollout continue past the test set's own range: seeded from the test
set's initial conditions (the oldest rows of `gpr_full_traj`, ~1243 ka BP), then rolled all
the way to the present using the full trajectory for insolation forcing and as the ground-truth
(+/-1 sigma) comparison for the whole span.

In [ ]:
# --- extended rollout: seed from the test set's own initial conditions and roll all the way to the
# present, using the full trajectory (obs_data/gpr_full_traj) for both the insolation forcing
# beyond the test window and as the ground-truth + 1-sigma comparison ---
full_traj_data = train_scaler.transform(np.column_stack([X, Y, Z, R]))
full_traj_std_raw = np.column_stack([
    obs_data['dO18_std'].to_numpy(dtype=float),
    obs_data['CO2_std'].to_numpy(dtype=float),
    obs_data['MgCa_std'].to_numpy(dtype=float),
])
# stds only get divided by the scaler's per-feature scale - no mean subtraction, that's for values not spreads
full_traj_std_scaled = full_traj_std_raw / train_scaler.scale_[:3]
full_traj_time = obs_data['Age[kaBP]'].to_numpy(dtype=float)

window_seed = test_data[:window_size, :]  # test set's own initial conditions, ~1243-1194 ka BP
total_steps = full_traj_data.shape[0] - window_size  # rolls all the way to age 0

ed = list(window_seed)
for t in tqdm(range(total_steps), desc="Extended Test Forecasting"):
    current_window = np.array(ed[-window_size:])
    Input = np.expand_dims(current_window, axis=0)
    nxt_increments = predict_step(Input)

    nxt_step = np.zeros(4)
    nxt_step[0:3] = current_window[-1, 0:3] + nxt_increments
    nxt_step[3] = full_traj_data[window_size + t, -1]

    ed.append(nxt_step)

emulator_long_test_data = np.array(ed)

In [ ]:
# PLOTTING ROLLOUT ON TEST SET
fig,axs = plt.subplots(3,1,figsize=(20,6),sharex=True)

mappings = ["X","Y","Z"]

for i in range(len(axs)):
    axs[i].plot(
        test_time[:emulator_test_data.shape[0]],
        emulator_test_data[:,i],
        color = color_ls[i][1],
        label="emulator data")

    axs[i].plot(
        test_time[:emulator_test_data.shape[0]],
        test_data[:emulator_test_data.shape[0],i],
        label="real scaled data",
        alpha = 0.4,
        linestyle = "dashed",
        color = color_ls[i][0])
    

    #axs[i].set_ylim(
    #    top = np.max(test_data[:emulator_test_data.shape[0],i]),
    #    bottom = np.min(test_data[:emulator_test_data.shape[0],i]))

    axs[i].set_title("$\mathcal{L}_{L_2}$"+f"= {mean_squared_error(test_data[window_size:,i],emulator_test_data[window_size:,i]):.3f}")
    axs[i].invert_xaxis()
    axs[i].set_ylabel(mappings[i])

#fig.legend()
axs[-1].set_xlabel("time [kyr]")
plt.tight_layout()
#plt.savefig(f"{exportpath}/plotTest_Rollout.pdf")
plt.show()

# PLOTTING EXTENDED ROLLOUT (1.2 Mya BP -> present) vs. full stitched ground truth
fig, axs = plt.subplots(3, 1, figsize=(20, 9), sharex=True)
mappings = ["X", "Y", "Z"]

n_plot = emulator_long_test_data.shape[0]

# naive baseline: trailing moving average of the ground truth itself (same window
# length the model's own input window uses). Note this isn't a fully fair forecast
# comparison - it always sees the real recent value at every point, unlike the
# autoregressive rollout, which only touches real data once, at the seed.
ma_window = window_size
moving_avg = np.column_stack([
    pd.Series(full_traj_data[:n_plot, i]).rolling(ma_window, min_periods=1).mean().to_numpy()
    for i in range(3)
])

for i in range(3):
    axs[i].plot(
        full_traj_time[:n_plot],
        emulator_long_test_data[:, i],
        color=color_ls[i][1],
        label="emulator data")

    axs[i].plot(
        full_traj_time[:n_plot],
        full_traj_data[:n_plot, i],
        color=color_ls[i][0],
        linestyle="dashed",
        alpha=0.6,
        label="ground truth (gpr_full_traj)")

    axs[i].fill_between(
        full_traj_time[:n_plot],
        full_traj_data[:n_plot, i] - full_traj_std_scaled[:n_plot, i],
        full_traj_data[:n_plot, i] + full_traj_std_scaled[:n_plot, i],
        color=color_ls[i][0], alpha=0.2, label="ground truth ±1σ")

    axs[i].plot(
        full_traj_time[:n_plot],
        moving_avg[:, i],
        color="black",
        alpha=0.7,
        linestyle=":",
        label=f"moving average ({ma_window} kyr)")

    axs[i].invert_xaxis()
    axs[i].set_ylabel(mappings[i])
    axs[i].legend(loc="upper right", fontsize=12)

    # per-variable error, seed window excluded (it trivially matches by construction) -
    # same RMSE/R² convention used in the "Full synthetic rollout" section
    rmse_i = np.sqrt(mean_squared_error(full_traj_data[window_size:n_plot, i], emulator_long_test_data[window_size:, i]))
    r2_i = r2_score(full_traj_data[window_size:n_plot, i], emulator_long_test_data[window_size:, i])
    ma_rmse_i = np.sqrt(mean_squared_error(full_traj_data[window_size:n_plot, i], moving_avg[window_size:n_plot, i]))
    axs[i].set_title(f"RMSE = {rmse_i:.3f}, R\u00b2 = {r2_i:.3f}  |  MA RMSE = {ma_rmse_i:.3f}")

axs[-1].set_xlabel("Age [ka BP]")
plt.tight_layout()
plt.show()

# overall error across all three state variables combined (X, Y, Z), seed window excluded
overall_rmse = np.sqrt(mean_squared_error(full_traj_data[window_size:n_plot, 0:3], emulator_long_test_data[window_size:, 0:3]))
overall_r2 = r2_score(full_traj_data[window_size:n_plot, 0:3], emulator_long_test_data[window_size:, 0:3])
overall_ma_rmse = np.sqrt(mean_squared_error(full_traj_data[window_size:n_plot, 0:3], moving_avg[window_size:n_plot, 0:3]))
print(f"Extended rollout overall error (X,Y,Z combined, seed window excluded): RMSE = {overall_rmse:.3f}, R\u00b2 = {overall_r2:.3f}")
print(f"Moving-average baseline overall error (X,Y,Z combined, seed window excluded): RMSE = {overall_ma_rmse:.3f}")

In [ ]:
if flag:
    # --- continue the rollout PAST the present (age 0), where gpr_full_traj/full_traj_data
    # has no more real data - use the Laskar-derived long_R/long_R_scaled as forcing instead
    # (see earlier discussion: insolation is a purely astronomical quantity, so unlike the
    # ice-core-derived X/Y/Z proxies it CAN be computed for the future) ---
    future_mask = years > 0
    future_R_scaled = long_R_scaled[future_mask]
    future_years = years[future_mask]

    ed = list(emulator_long_test_data[-window_size:])  # continue from where the extended rollout left off
    for t in tqdm(range(len(future_R_scaled)), desc="Future Forecasting"):
        current_window = np.array(ed[-window_size:])
        Input = np.expand_dims(current_window, axis=0)
        nxt_increments = predict_step(Input)

        nxt_step = np.zeros(4)
        nxt_step[0:3] = current_window[-1, 0:3] + nxt_increments
        nxt_step[3] = future_R_scaled[t]

        ed.append(nxt_step)

    emulator_future_data = np.array(ed)[window_size:]  # drop the duplicated seed window

    # stitch historical (1243 ka BP -> present) and future (present -> +10 Ma) rollouts into
    # one continuous trajectory, on a single "years from present" axis (negative = past,
    # matching the `years`/long_R convention - opposite sign from Age[kaBP])
    hist_years = -full_traj_time[:n_plot]
    combined_time = np.concatenate([hist_years, future_years])
    combined_emulator = np.concatenate([emulator_long_test_data[:n_plot], emulator_future_data])

    # PLOTTING ROLLOUT CONTINUED PAST PRESENT, WITH INSOLATION FORCING
    fig, axs = plt.subplots(4, 1, figsize=(20, 12), sharex=True)
    mappings = ["X", "Y", "Z"]

    for i in range(3):
        axs[i].plot(
            combined_time,
            combined_emulator[:, i],
            color=color_ls[i][1],
            label="emulator data")

        # ground truth only exists up to the present (age 0) - nothing to compare against
        # for the future portion
        axs[i].plot(
            hist_years,
            full_traj_data[:n_plot, i],
            color=color_ls[i][0],
            linestyle="dashed",
            alpha=0.6,
            label="ground truth (gpr_full_traj)")

        axs[i].axvline(x=0, color="gray", linestyle=":", alpha=0.6)
        axs[i].set_ylabel(mappings[i])
        axs[i].legend(loc="upper right", fontsize=12)

    # fourth panel: the insolation (ISI) forcing driving the rollout over the same span
    combined_R = np.concatenate([long_R[years <= 0], long_R[future_mask]])
    axs[3].plot(combined_time, combined_R[:11242], color="tab:orange", label="ISI (65N, thresh0)")
    axs[3].axvline(x=0, color="gray", linestyle=":", alpha=0.6)
    axs[3].set_ylabel("ISI [GJ/m$^2$]")
    axs[3].legend(loc="upper right", fontsize=12)

    axs[-1].set_xlabel("Time from present [kyr] (negative = past)")
    plt.tight_layout()
    plt.show()


## Rollout on training set (SM90 train+validation timeline)

A reference rollout over the clean synthetic record, independent of the observational test
above — useful as a baseline to compare against the observational rollout's fidelity.

In [ ]:
# PLOTTING FULL ROLLOUT OF EMULATOR AND NUMERICAL MODEL
# note: this rollout is over the synthetic `data` timeline (train+validation) only.
# the obs test set lives on its own independent timeline (obs_time) and is not a
# prefix of `data`, so it cannot be stitched onto this rollout the way the
# corrupted-test-set version could.
test_std = tf.math.reduce_std(test_data[window_size:,0:3],axis = 0)
rollout_test_std = tf.math.reduce_std(emulator_test_data[window_size:,0:3],axis = 0)
print("C_2 on the test set:")
print( tf.reduce_mean(tf.abs(1.0 - (rollout_test_std / (test_std + 1e-6)))))


In [ ]:
data_scaled = train_scaler.transform(fulldata)
tv_data_scaled = train_scaler.transform(trainval)

full_rollout = list(tv_data_scaled[:window_size,:])

for t in tqdm(range(tv_data_scaled.shape[0]-window_size), desc="Full Forecasting"):

    nxt_step = np.zeros(4)
    # Grab the window directly from the main array to ensure it's always up to date
    current_window = np.array(full_rollout[-window_size:])

    # Predict
    Input = np.expand_dims(current_window, axis=0)

    nxt_increments = predict_step(Input)

    nxt_step[0:3] = current_window[-1,0:3] + nxt_increments

    # write new insolation from data to next step
    nxt_step[3] = tv_data_scaled[window_size+t,-1]

    full_rollout.append(nxt_step)

full_rollout = np.array(full_rollout)


In [ ]:
fig,axs = plt.subplots(3,1,figsize=(20,6),sharex=True)

mappings = ["X","Y","Z"]

for i in range(len(axs)):
    axs[i].plot(
        trainval_time[:full_rollout.shape[0]],
        full_rollout[:,i],
        color = color_ls[i][1],
        label="emulator data")

    axs[i].plot(
        trainval_time[:full_rollout.shape[0]],
        tv_data_scaled[:full_rollout.shape[0],i],
        label="real scaled data",
        alpha = 0.4,
        linestyle = "dashed",
        color = color_ls[i][0])

    axs[i].set_ylim(
        top = np.max(tv_data_scaled[:full_rollout.shape[0],i]),
        bottom = np.min(tv_data_scaled[:full_rollout.shape[0],i]))

    axs[i].set_title(f"RMSE = {np.sqrt(mean_squared_error(tv_data_scaled[:full_rollout.shape[0],i],full_rollout[:,i])):.3f}, R² = {r2_score(tv_data_scaled[:full_rollout.shape[0],i],full_rollout[:,i]):.3f}")

    axs[i].invert_xaxis()
    axs[i].set_ylabel(mappings[i])

#fig.legend()
axs[-1].set_xlabel("time [kyr]")
plt.tight_layout()
#plt.savefig(f"{exportpath}/Full_rollout.pdf")
plt.show()


## Phase-space comparison

In [ ]:
plt.gca()

# PLOTTING PHASE SPACE OF FULL ROLLOUT NN AND NUMERICAL MODEL
fig, axs = plt.subplots(1,3,figsize=(18,6))


axs[0].plot(
    full_traj_data[:,0],
    full_traj_data[:,1],
    color = "skyblue",
    linestyle = "dashed",
    label="scaled data")

axs[0].plot(
    emulator_long_test_data[:,0],
    emulator_long_test_data[:,1],
    color = "steelblue",
    label="emulator data")

axs[0].set_xlabel("X")
axs[0].set_ylabel("Y")


axs[1].plot(
    full_traj_data[:,0],
    full_traj_data[:,2],
    color = "skyblue",
    linestyle = "dashed")

axs[1].plot(
    emulator_long_test_data[:,0],
    emulator_long_test_data[:,2],
    color = "steelblue")


axs[1].set_xlabel("X")
axs[1].set_ylabel("Z")

axs[2].plot(
    full_traj_data[:,1],
    full_traj_data[:,2],
    color = "skyblue",
    linestyle = "dashed")

axs[2].plot(
    emulator_long_test_data[:,1],
    emulator_long_test_data[:,2],
    color = "steelblue")

axs[2].set_xlabel("Y")
axs[2].set_ylabel("Z")

plt.tight_layout(rect=[0, 0.08, 1, 1])
handles, labels = axs[0].get_legend_handles_labels()

fig.legend(handles, labels,
           loc='lower center',
           bbox_to_anchor=(0.5, -0.005),
           ncol=3,
           fontsize=20,
           frameon=True,
           facecolor='white',
           edgecolor='gray')

#plt.savefig(f"{exportpath}/Phase_Space_Full_Rollout.pdf",bbox_inches="tight")
plt.show()



In [ ]:
if flag:
    plt.gca()

    # PLOTTING PHASE SPACE OF FULL ROLLOUT NN AND NUMERICAL MODEL
    fig, axs = plt.subplots(1,3,figsize=(18,6))


    axs[0].plot(
        full_traj_data[:,0],
        full_traj_data[:,1],
        color = "skyblue",
        linestyle = "dashed",
        label="scaled data")

    axs[0].plot(
        combined_emulator[:,0],
        combined_emulator[:,1],
        color = "steelblue",
        label="emulator data")

    axs[0].set_xlabel("X")
    axs[0].set_ylabel("Y")


    axs[1].plot(
        full_traj_data[:,0],
        full_traj_data[:,2],
        color = "skyblue",
        linestyle = "dashed")

    axs[1].plot(
        combined_emulator[:,0],
        combined_emulator[:,2],
        color = "steelblue")


    axs[1].set_xlabel("X")
    axs[1].set_ylabel("Z")

    axs[2].plot(
        full_traj_data[:,1],
        full_traj_data[:,2],
        color = "skyblue",
        linestyle = "dashed")

    axs[2].plot(
        combined_emulator[:,1],
        combined_emulator[:,2],
        color = "steelblue")

    axs[2].set_xlabel("Y")
    axs[2].set_ylabel("Z")

    plt.tight_layout(rect=[0, 0.08, 1, 1])
    handles, labels = axs[0].get_legend_handles_labels()

    fig.legend(handles, labels,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.005),
            ncol=3,
            fontsize=20,
            frameon=True,
            facecolor='white',
            edgecolor='gray')

    #plt.savefig(f"{exportpath}/Phase_Space_Full_Rollout.pdf",bbox_inches="tight")
    plt.show()



## Fourier spectrum comparison

### Fourier using Multitaper library

In [ ]:
import multitaper as mt
from scipy import stats

def multitaper_spectrum(x, dt, nw=2):
    """Multitaper spectrum + classical chi-squared CI + Thomson F-test significance mask."""
    kspec = int(2*nw - 1)   # number of tapers - more tapers = less variance, less resolution
    psd = mt.MTSpec(x, nw=nw, kspec=kspec, dt=dt, iadapt=0)

    freq, spec = psd.rspec()
    freq = freq.squeeze()
    spec = spec.squeeze()

    one_sided = psd.freq.squeeze() >= 0

    # jackspec is either buggy or I (and Claude) can't figure it out - classical CI instead
    dof = 2 * kspec
    ci_lower = spec * dof / stats.chi2.ppf(0.975, dof)
    ci_upper = spec * dof / stats.chi2.ppf(0.025, dof)

    F, _ = psd.ftest()
    F = F.squeeze()[one_sided]
    f_crit = stats.f.ppf(0.95, 2, 2*kspec - 2)   # df1=2, df2=2K-2 for the harmonic F-test
    significant = F > f_crit

    return freq, spec, ci_lower, ci_upper, significant

dt = timestep
nw = 2
SPEC_VAR = 0   # 0 = X, 1 = Y, 2 = Z (same variable for ground truth and emulator)

# label -> (series, color)
datasets = {
    "gt":   ("Ground truth",                data_scaled[:, SPEC_VAR],             '#D6195E'),
    #"tv":   ("Training/validation rollout", full_rollout[:, SPEC_VAR],            '#1E88E5'),
    "test": ("Test rollout (long)",         emulator_long_test_data[:, SPEC_VAR], '#43A047'),
}

results = {}
fig, ax = plt.subplots(figsize=(12, 6))

for key, (label, x, color) in datasets.items():
    freq, spec, ci_lower, ci_upper, significant = multitaper_spectrum(x, dt, nw)
    results[key] = {"freq": freq, "spec": spec, "ci_lower": ci_lower, "ci_upper": ci_upper, "significant": significant}

    ax.plot(freq, spec, color=color, label=f"{label} spectrum")
    ax.fill_between(freq, ci_lower, ci_upper, color=color, alpha=0.15)
    ax.scatter(freq[significant], spec[significant], color=color, edgecolor="black", s=20, zorder=5)

    print(f"{label} significant periods (kyr):", 1 / freq[significant])

ymin = min(r["spec"].min() for r in results.values())
ymax = max(r["spec"].max() for r in results.values())
ax.vlines(1/100, ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", label="100 kyr cycle", color="darkblue")
ax.vlines(1/41,  ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", label="41 kyr cycle",  color="slateblue")

ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("frequency [1/kyr]")
ax.set_ylabel("power")
ax.set_title(f"Multitaper spectra ({mappings[SPEC_VAR]}): ground truth vs. autoregressive rollouts")
ax.legend()
plt.tight_layout()
plt.show()

# unpacked, suffixed variables kept for any later cells that reference them directly
freq_gt, spec_gt, ci_lower_gt, ci_upper_gt, significant_gt = (results["gt"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))
#freq_tv, spec_tv, ci_lower_tv, ci_upper_tv, significant_tv = (results["tv"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))
freq_test, spec_test, ci_lower_test, ci_upper_test, significant_test = (results["test"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))

## Extras

### Export

In [ ]:
n = emulator_long_test_data.shape[0]

# inverse-transform back to physical units using the same scaler the model trained on
unscaled = train_scaler.inverse_transform(emulator_long_test_data)

export_df = pd.DataFrame({
    "Age[kaBP]": full_traj_time[:n],
    "X_scaled": emulator_long_test_data[:, 0],
    "Y_scaled": emulator_long_test_data[:, 1],
    "Z_scaled": emulator_long_test_data[:, 2],
    "Insolation_scaled": emulator_long_test_data[:, 3],
    "dO18_LR04": unscaled[:, 0],
    "CO2": unscaled[:, 1],
    "MgCa": unscaled[:, 2],
    "Insolation[W/m2]": unscaled[:, 3],
})

export_df.to_csv(rc.MODEL_OUTPUT_DIR / f"{cfg['run_tag']}_hMSE_test_rollout.csv", index=False)

In [ ]:
if flag:
    # --- export the stitched rollout (1243 ka BP -> +10 Ma), same convention as the
    # emulator_long_test_data export above: scaled + inverse-transformed physical
    # columns, one row per kyr step ---
    unscaled_combined = train_scaler.inverse_transform(combined_emulator)

    export_future_df = pd.DataFrame({
        "Time_from_present[kyr]": combined_time,  # negative = past, positive = future (matches `years`)
        "X_scaled": combined_emulator[:, 0],
        "Y_scaled": combined_emulator[:, 1],
        "Z_scaled": combined_emulator[:, 2],
        "Insolation_scaled": combined_emulator[:, 3],
        "dO18_LR04": unscaled_combined[:, 0],
        "CO2": unscaled_combined[:, 1],
        "MgCa": unscaled_combined[:, 2],
        "Insolation[GJ/m2]": unscaled_combined[:, 3],
    })

    export_future_df.to_csv(rc.MODEL_OUTPUT_DIR / f"{cfg['run_tag']}_future_rollout_10Ma.csv", index=False)

    export_future_df


### Archive

In [ ]:
## Old frequency spectra
#PLOT FOURIER TRANSORM SPECTRUM OF EMULATOR AND SM90 DATA
# calculate spectrum
fourier_obs = np.fft.fft(data_scaled[:,0])
fft_coeff_obs = np.abs(fourier_obs)
fft_freq_obs = np.fft.fftfreq(n=len(data_scaled[:,0]),d=timestep)
fft_freq_mask_obs= fft_freq_obs >= 0

fourier_Emu = np.fft.fft(full_rollout[:,0])
fft_coeff_Emu = np.abs(fourier_Emu)
fft_freq_Emu = np.fft.fftfreq(n=len(full_rollout[:,0]),d=timestep)
fft_freq_mask_Emu = fft_freq_Emu >= 0

fourier_Emu_long = np.fft.fft(emulator_long_test_data[:,0])
fft_coeff_Emu_long = np.abs(fourier_Emu_long)
fft_freq_Emu_long = np.fft.fftfreq(n=len(emulator_long_test_data[:,0]),d=timestep)
fft_freq_mask_Emu_long = fft_freq_Emu_long >= 0

print(f"max. emulator frequency: {fft_freq_Emu[np.argmax(fft_coeff_Emu[fft_freq_mask_Emu])]} ({fft_freq_Emu[np.argmax(fft_coeff_Emu[fft_freq_mask_Emu])]**-1})")
print(f"max. observation frequency: {fft_freq_obs[np.argmax(fft_coeff_obs[fft_freq_mask_obs])]} ({fft_freq_obs[np.argmax(fft_coeff_obs[fft_freq_mask_obs])]**-1})")
print(f"max. long emulation frequency: {fft_freq_Emu_long[np.argmax(fft_coeff_Emu_long[fft_freq_mask_Emu_long])]} ({fft_freq_Emu_long[np.argmax(fft_coeff_Emu_long[fft_freq_mask_Emu_long])]**-1})")

fig, axs = plt.subplots(figsize=(12,6))

axs.plot(fft_freq_obs[fft_freq_mask_obs],fft_coeff_obs[fft_freq_mask_obs],label="Obs")
axs.plot(fft_freq_Emu[fft_freq_mask_Emu],fft_coeff_Emu[fft_freq_mask_Emu],label="SR-LPNE")
axs.plot(fft_freq_Emu_long[fft_freq_mask_Emu_long],fft_coeff_Emu_long[fft_freq_mask_Emu_long],label="SR-LPNE (long)")
axs.vlines(1/100,ymin=np.min(fft_coeff_obs[fft_freq_mask_obs]),ymax=np.max(fft_coeff_obs[fft_freq_mask_obs]),alpha = 0.5,linestyle="dashed",label="100 kyr cycle",color="darkblue")
axs.vlines(1/41,ymin=np.min(fft_coeff_obs[fft_freq_mask_obs]),ymax=np.max(fft_coeff_obs[fft_freq_mask_obs]),alpha = 0.5,linestyle="dashed",label="41 kyr cycle",color="slateblue")
axs.set_yscale("log")
axs.set_xscale("log")
axs.set_ylabel("coefficents")
axs.set_xlabel("frequencies 1/kyr")

fig.legend(loc="upper right")
plt.tight_layout()
#plt.savefig(f"{exportpath}/Spectrum_Emulator.pdf")
plt.show()

fourier_SM90_data = np.zeros((len(fft_freq_obs[fft_freq_mask_obs]),2))
fourier_SM90_data[:,0] = fft_freq_obs[fft_freq_mask_obs]
fourier_SM90_data[:,1] = fft_freq_obs[fft_freq_mask_obs]

fourier_Emu_data = np.zeros((len(fft_freq_Emu[fft_freq_mask_Emu]),2))
fourier_Emu_data[:,0] = fft_freq_Emu[fft_freq_mask_Emu]
fourier_Emu_data[:,1] = fft_coeff_Emu[fft_freq_mask_Emu]

#np.savetxt(f"{exportpath}/fourier_data_SM90.csv",fourier_SM90_data,delimiter=",")
#np.savetxt(f"{exportpath}/fourier_data_Emu.csv",fourier_Emu_data,delimiter=",")
